In [112]:
import pandas as pd
import json
import matplotlib.pyplot as plt
import numpy as np

In [113]:
experiments = [
    {"id": "exp_20260602_170520", "name": "Openwhisk"},
    {"id": "exp_20260602_154435", "name": "NMIG"},
    {"id": "exp_20260910_221952", "name": "Profiler"},
    {"id": "exp_20260910_181356", "name": "UCB"},
]

In [114]:
# import pandas as pd
# import json

def parse_json_safe(val):
    if pd.isna(val) or val == "None":
        return None
    if isinstance(val, dict):  # Already a dict
        return val
    try:
        return json.loads(val)
    except Exception:
        return None

def extract_name(res,name):
    if isinstance(res, dict):
        return res.get(name)
    return None




def extract_error_message(res):
    if isinstance(res, dict):
        resp = res.get("response", {})
        if not resp.get("success", True):
            return resp.get("result", {}).get("error")
    return None

def has_cuda_error(msg):
    if isinstance(msg, str):
        return "cuda" in msg.lower() or "cudnn" in msg.lower()
    return False

def extract_success(res):
    if isinstance(res, dict):
        return res.get("response", {}).get("success", True)
    return False



In [115]:
# plt.figure(figsize=(4, 3.5))
results = {}
df_res = {}

def pick_particular_column(rp,col="initTime"):
    # rp is expected to be a dict with key 'annotations' that is a list of dicts
    if not isinstance(rp, dict):
        return 0
    items = rp.get("annotations", []) or []
    for it in items:
        if isinstance(it, dict) and it.get("key") == col:
            return (it.get("value", 0))/1000
    return 0

    
for exp in experiments:
    # load the CSV for this experiment
    path = f"../results/{exp['id']}/results_updated.csv"
    df = pd.read_csv(path)

    # parse JSON and compute latency
    df["result_parsed"] = df["result"].apply(parse_json_safe)
    df["name"] = df["result_parsed"].apply(lambda res: extract_name(res, "name"))
    df['name'] = df['name'].str.replace(r'_p$', '', regex=True)
    df["start"] = df["result_parsed"].apply(lambda res: extract_name(res, "start"))
    df["end"] = df["result_parsed"].apply(lambda res: extract_name(res, "end"))
    df['latency'] = (df['end'] - df['start'])/1000
    df["error_message"] = df["result_parsed"].apply(extract_error_message)
    df["has_cuda_or_cudnn_error"] = df["error_message"].apply(has_cuda_error)
    df["success"] = df["result_parsed"].apply(extract_success)
    df["error"] = df["result_parsed"].isna() | (~df["success"])
    df["initTime"] = df["result_parsed"].apply(lambda x: pick_particular_column(x, col="initTime"))
    df["waitTime"] = df["result_parsed"].apply(lambda x: pick_particular_column(x, col="waitTime"))
    

    df_res[exp['name']] = df
    # error_counts = df['error'].value_counts()
    # results[exp['name']] = {}
    # results[exp['name']]['true_val'] = error_counts.get(True, 0)
    # results[exp['name']]['false_val'] = error_counts.get(False, 0)
    # results[exp['name']]['false_val'] = df['latency'].mean()
    # df_valid = df[df["error"] != True]
    # avg_latency_per_name_dict = df_valid.groupby("name")["latency"].mean().to_dict()
    # print(avg_latency_per_name_dict)
    # df_res[exp['
    print(exp)
    
   
    # df = df[df['latency'] < 300]

print(results)

{'id': 'exp_20260602_170520', 'name': 'Openwhisk'}
{'id': 'exp_20260602_154435', 'name': 'NMIG'}
{'id': 'exp_20260910_221952', 'name': 'Profiler'}
{'id': 'exp_20260910_181356', 'name': 'UCB'}
{}


In [116]:

experiments = [
    {'id': 'exp_20250626_205804', 'name': 'OpenWhisk'},
    {'id': 'exp_20260915_152838', 'name': 'Process'},
    {'id': 'exp_20260813_234749', 'name': 'UCB-1'},
    {'id': 'exp_20250627_042542', 'name': 'NMIG'},
]

results = []

for exp in experiments:
    sf = pd.read_csv(f"../results/{exp['id']}/gpu_usage.csv")

    # Sum across all containers per timestamp
    df_agg = sf.groupby('Timestamp')['GPU_Memory_MB'].sum().reset_index()
    df_agg = df_agg.sort_values('Timestamp')

    # Time delta in seconds (timestamps in ms)
    df_agg['time_delta_s'] = df_agg['Timestamp'].diff().fillna(0) / 1000

    # GB·s per row
    df_agg['gb_s'] = (df_agg['GPU_Memory_MB'] / 1024) * df_agg['time_delta_s']

    total_gb_s = df_agg['gb_s'].sum()
    mean_mb    = df_agg['GPU_Memory_MB'].mean()
    peak_mb    = df_agg['GPU_Memory_MB'].max()

    results.append({
        'Configuration': exp['name'],
        'GPU mem-time (GB·s)': round(total_gb_s, 2),
        'Mean resident (MB)':  round(mean_mb, 2),
        'Peak (MB)':           round(peak_mb, 2),
    })

df_results = pd.DataFrame(results)

# Compute reduction % relative to OpenWhisk
baseline_gbs  = df_results.loc[df_results['Configuration'] == 'OpenWhisk', 'GPU mem-time (GB·s)'].values[0]
baseline_mean = df_results.loc[df_results['Configuration'] == 'OpenWhisk', 'Mean resident (MB)'].values[0]

df_results['GPU mem reduction (%)'] = df_results['GPU mem-time (GB·s)'].apply(
    lambda x: round((1 - x / baseline_gbs) * 100, 1)
)
df_results['Mean mem reduction (%)'] = df_results['Mean resident (MB)'].apply(
    lambda x: round((1 - x / baseline_mean) * 100, 1)
)

print(df_results.to_string(index=False))

Configuration  GPU mem-time (GB·s)  Mean resident (MB)  Peak (MB)  GPU mem reduction (%)  Mean mem reduction (%)
    OpenWhisk             97239.33             6648.89    26508.0                    0.0                     0.0
      Process             25342.99             3704.53    10574.0                   73.9                    44.3
        UCB-1             23849.11             3088.89     6272.0                   75.5                    53.5
         NMIG             11681.20              996.67    12114.0                   88.0                    85.0


In [117]:
experiments = [
    {"id": "exp_20250626_205804", "name": "Openwhisk"},
        {'id': 'exp_20260917_123621', 'name': 'Unlock'},
        {'id': 'exp_20260915_152838', 'name': 'Process'},
        {"id": "exp_20260910_221952", "name": "Profiler"},
        {"id": "exp_20250627_042542", "name": "NMIG"},
    
]

T_TARGET = 4 * 3600
summary = {}

for exp in experiments:
    fn = pd.read_csv(f"../results/{exp['id']}/results_updated.csv")
    df = pd.read_csv(f"../results/{exp['id']}/gpu_usage.csv")

    initial_time_ms = fn['timestamp'].iloc[0]
    df['second_level_datetime'] = pd.to_datetime(df['Timestamp'], unit='ms').dt.floor('s')

    df_avg = (df.groupby(['ContainerID', 'second_level_datetime'])['GPU_Memory_MB']
                .mean().reset_index())
    mem_per_second = (df_avg.groupby('second_level_datetime')['GPU_Memory_MB']
                            .sum().reset_index(name='total_mem_mb'))

    initial_dt = pd.to_datetime(initial_time_ms, unit='ms')
    mem_per_second['rel_s'] = ((mem_per_second['second_level_datetime'] - initial_dt)
                               .dt.total_seconds().astype(int))
    mem_per_second = mem_per_second[mem_per_second['rel_s'] >= 0].reset_index(drop=True)

    full = pd.DataFrame({'rel_s': range(T_TARGET + 1)})
    m = full.merge(mem_per_second[['rel_s', 'total_mem_mb']], on='rel_s', how='left')
    m['total_mem_mb'] = m['total_mem_mb'].fillna(0)

    gpu_gb_s      = m['total_mem_mb'].sum() / 1024.0
    mean_mb       = m['total_mem_mb'].mean()
    peak_mb       = m['total_mem_mb'].max()

    summary[exp['name']] = {
        'GPU mem-time (GB·s)': round(gpu_gb_s, 2),
        'Mean resident (MB)':  round(mean_mb, 2),
        'Peak (MB)':           round(peak_mb, 2),
    }

df_results = pd.DataFrame(summary).T

# Reduction relative to OpenWhisk
baseline = summary['Openwhisk']
for name, row in summary.items():
    if name == 'Openwhisk':
        df_results.loc[name, 'GPU mem reduction (%)'] = 0.0
        df_results.loc[name, 'Mean mem reduction (%)'] = 0.0
    else:
        df_results.loc[name, 'GPU mem reduction (%)'] = round(
            (1 - row['GPU mem-time (GB·s)'] / baseline['GPU mem-time (GB·s)']) * 100, 1)
        df_results.loc[name, 'Mean mem reduction (%)'] = round(
            (1 - row['Mean resident (MB)'] / baseline['Mean resident (MB)']) * 100, 1)

print(df_results.to_string())

           GPU mem-time (GB·s)  Mean resident (MB)  Peak (MB)  GPU mem reduction (%)  Mean mem reduction (%)
Openwhisk             44965.13             3197.30    13254.0                    0.0                     0.0
Unlock                34425.18             2447.84     4067.0                   23.4                    23.4
Process               10200.32              725.31     5464.0                   77.3                    77.3
Profiler               9640.59              685.51     6260.0                   78.6                    78.6
NMIG                    447.91               31.85     6623.0                   99.0                    99.0


In [118]:
#Similar dataset

experiments = [
    {"id": "exp_20250625_162425", "name": "Openwhisk"},
        {'id': 'exp_20260922_131116', 'name': 'Unlock'},
        {'id': 'exp_20260922_173728', 'name': 'Process'},
        {"id": "exp_20260922_222414", "name": "Profiler"},
        {"id": "exp_20250625_113106", "name": "NMIG"},
]

T_TARGET = 4 * 3600
summary = {}

for exp in experiments:
    fn = pd.read_csv(f"../results/{exp['id']}/results_updated.csv")
    df = pd.read_csv(f"../results/{exp['id']}/gpu_usage.csv")

    initial_time_ms = fn['timestamp'].iloc[0]
    df['second_level_datetime'] = pd.to_datetime(df['Timestamp'], unit='ms').dt.floor('s')

    df_avg = (df.groupby(['ContainerID', 'second_level_datetime'])['GPU_Memory_MB']
                .mean().reset_index())
    mem_per_second = (df_avg.groupby('second_level_datetime')['GPU_Memory_MB']
                            .sum().reset_index(name='total_mem_mb'))

    initial_dt = pd.to_datetime(initial_time_ms, unit='ms')
    mem_per_second['rel_s'] = ((mem_per_second['second_level_datetime'] - initial_dt)
                               .dt.total_seconds().astype(int))
    mem_per_second = mem_per_second[mem_per_second['rel_s'] >= 0].reset_index(drop=True)

    full = pd.DataFrame({'rel_s': range(T_TARGET + 1)})
    m = full.merge(mem_per_second[['rel_s', 'total_mem_mb']], on='rel_s', how='left')
    m['total_mem_mb'] = m['total_mem_mb'].fillna(0)

    gpu_gb_s      = m['total_mem_mb'].sum() / 1024.0
    mean_mb       = m['total_mem_mb'].mean()
    peak_mb       = m['total_mem_mb'].max()

    summary[exp['name']] = {
        'GPU mem-time (GB·s)': round(gpu_gb_s, 2),
        'Mean resident (MB)':  round(mean_mb, 2),
        'Peak (MB)':           round(peak_mb, 2),
    }

df_results = pd.DataFrame(summary).T

# Reduction relative to OpenWhisk
baseline = summary['Openwhisk']
for name, row in summary.items():
    if name == 'Openwhisk':
        df_results.loc[name, 'GPU mem reduction (%)'] = 0.0
        df_results.loc[name, 'Mean mem reduction (%)'] = 0.0
    else:
        df_results.loc[name, 'GPU mem reduction (%)'] = round(
            (1 - row['GPU mem-time (GB·s)'] / baseline['GPU mem-time (GB·s)']) * 100, 1)
        df_results.loc[name, 'Mean mem reduction (%)'] = round(
            (1 - row['Mean resident (MB)'] / baseline['Mean resident (MB)']) * 100, 1)

print(df_results.to_string())

           GPU mem-time (GB·s)  Mean resident (MB)  Peak (MB)  GPU mem reduction (%)  Mean mem reduction (%)
Openwhisk             41254.20             2933.43     6162.0                    0.0                     0.0
Unlock                34420.28             2447.49     4130.0                   16.6                    16.6
Process                6710.61              477.17     3152.0                   83.7                    83.7
Profiler               6731.52              478.65     3140.0                   83.7                    83.7
NMIG                    576.53               40.99     2551.0                   98.6                    98.6


In [119]:
#Brusty

experiments = [
      {"id": "exp_20250625_014444", "name": "Openwhisk"},
        {'id': 'exp_20260923_212417', 'name': 'Unlock'},
        {'id': 'exp_20260923_092219', 'name': 'Process'},
        {"id": "exp_20260923_161047", "name": "Profiler"},
        {"id": "exp_20250625_071954", "name": "NMIG"}
]

T_TARGET = 4 * 3600
summary = {}

for exp in experiments:
    fn = pd.read_csv(f"../results/{exp['id']}/results_updated.csv")
    df = pd.read_csv(f"../results/{exp['id']}/gpu_usage.csv")

    initial_time_ms = fn['timestamp'].iloc[0]
    df['second_level_datetime'] = pd.to_datetime(df['Timestamp'], unit='ms').dt.floor('s')

    df_avg = (df.groupby(['ContainerID', 'second_level_datetime'])['GPU_Memory_MB']
                .mean().reset_index())
    mem_per_second = (df_avg.groupby('second_level_datetime')['GPU_Memory_MB']
                            .sum().reset_index(name='total_mem_mb'))

    initial_dt = pd.to_datetime(initial_time_ms, unit='ms')
    mem_per_second['rel_s'] = ((mem_per_second['second_level_datetime'] - initial_dt)
                               .dt.total_seconds().astype(int))
    mem_per_second = mem_per_second[mem_per_second['rel_s'] >= 0].reset_index(drop=True)

    full = pd.DataFrame({'rel_s': range(T_TARGET + 1)})
    m = full.merge(mem_per_second[['rel_s', 'total_mem_mb']], on='rel_s', how='left')
    m['total_mem_mb'] = m['total_mem_mb'].fillna(0)

    gpu_gb_s      = m['total_mem_mb'].sum() / 1024.0
    mean_mb       = m['total_mem_mb'].mean()
    peak_mb       = m['total_mem_mb'].max()

    summary[exp['name']] = {
        'GPU mem-time (GB·s)': round(gpu_gb_s, 2),
        'Mean resident (MB)':  round(mean_mb, 2),
        'Peak (MB)':           round(peak_mb, 2),
    }

df_results = pd.DataFrame(summary).T

# Reduction relative to OpenWhisk
baseline = summary['Openwhisk']
for name, row in summary.items():
    if name == 'Openwhisk':
        df_results.loc[name, 'GPU mem reduction (%)'] = 0.0
        df_results.loc[name, 'Mean mem reduction (%)'] = 0.0
    else:
        df_results.loc[name, 'GPU mem reduction (%)'] = round(
            (1 - row['GPU mem-time (GB·s)'] / baseline['GPU mem-time (GB·s)']) * 100, 1)
        df_results.loc[name, 'Mean mem reduction (%)'] = round(
            (1 - row['Mean resident (MB)'] / baseline['Mean resident (MB)']) * 100, 1)

print(df_results.to_string())

           GPU mem-time (GB·s)  Mean resident (MB)  Peak (MB)  GPU mem reduction (%)  Mean mem reduction (%)
Openwhisk             98869.66             7030.24    22910.0                    0.0                     0.0
Unlock                39163.37             2784.76    10976.0                   60.4                    60.4
Process               30161.84             2144.69    10976.0                   69.5                    69.5
Profiler              30084.04             2139.16    10976.0                   69.6                    69.6
NMIG                    529.87               37.68    11071.0                   99.5                    99.5


In [111]:
import pandas as pd

T_TARGET = 4 * 3600

datasets = {
    "Bursty": [
        {"id": "exp_20250625_014444", "name": "Openwhisk"},
        {'id': 'exp_20260923_212417', 'name': 'Unlock'},
        {'id': 'exp_20260923_092219', 'name': 'Process'},
        {"id": "exp_20260923_161047", "name": "Profiler"},
        {"id": "exp_20250625_071954", "name": "NMIG"},
    ],
    "Normal": [
        {"id": "exp_20250626_205804", "name": "Openwhisk"},
        {'id': 'exp_20260917_123621', 'name': 'Unlock'},
        {'id': 'exp_20260915_152838', 'name': 'Process'},
        {"id": "exp_20260910_221952", "name": "Profiler"},
        {"id": "exp_20250627_042542", "name": "NMIG"},
    ],
    "Similar": [
        {"id": "exp_20250625_162425", "name": "Openwhisk"},
        {'id': 'exp_20260922_131116', 'name': 'Unlock'},
        {'id': 'exp_20260922_173728', 'name': 'Process'},
        {"id": "exp_20260922_222414", "name": "Profiler"},
        {"id": "exp_20250625_113106", "name": "NMIG"},
    ],
}

all_results = []

for dataset_name, experiments in datasets.items():
    for exp in experiments:
        fn = pd.read_csv(f"../results/{exp['id']}/results_updated.csv")
        df = pd.read_csv(f"../results/{exp['id']}/gpu_usage.csv")

        initial_time_ms = fn['timestamp'].iloc[0]
        initial_dt = pd.to_datetime(initial_time_ms, unit='ms')

        df['second_dt'] = pd.to_datetime(df['Timestamp'], unit='ms').dt.floor('s')

        # Mean per container per second, then sum across containers
        df_avg = (df.groupby(['ContainerID', 'second_dt'])['GPU_Memory_MB']
                    .mean().reset_index())
        mem_per_second = (df_avg.groupby('second_dt')['GPU_Memory_MB']
                                .sum().reset_index(name='total_mem_mb'))

        mem_per_second['rel_s'] = (
            (mem_per_second['second_dt'] - initial_dt)
            .dt.total_seconds().astype(int)
        )
        mem_per_second = mem_per_second[mem_per_second['rel_s'] >= 0].reset_index(drop=True)

        # Uniform 4-hour grid, zeros where no data
        full = pd.DataFrame({'rel_s': range(T_TARGET + 1)})
        m = full.merge(mem_per_second[['rel_s', 'total_mem_mb']],
                       on='rel_s', how='left')
        m['total_mem_mb'] = m['total_mem_mb'].fillna(0)

        all_results.append({
            'Dataset':             dataset_name,
            'Configuration':       exp['name'],
            'GPU mem-time (GB·s)': round(m['total_mem_mb'].sum() / 1024.0, 2),
            'Mean resident (MB)':  round(m['total_mem_mb'].mean(), 2),
            'Peak (MB)':           round(m['total_mem_mb'].max(), 2),
        })

df_results = pd.DataFrame(all_results)

# Compute reduction % relative to Openwhisk within each dataset
def add_reductions(df):
    baseline = df[df['Configuration'] == 'Openwhisk'].iloc[0]
    df = df.copy()
    df['GPU mem reduction (%)'] = df['GPU mem-time (GB·s)'].apply(
        lambda x: round((1 - x / baseline['GPU mem-time (GB·s)']) * 100, 1)
    )
    df['Mean mem reduction (%)'] = df['Mean resident (MB)'].apply(
        lambda x: round((1 - x / baseline['Mean resident (MB)']) * 100, 1)
    )
    return df

df_results = df_results.groupby('Dataset', group_keys=False).apply(add_reductions)

# Print grouped by dataset
for dataset_name, group in df_results.groupby('Dataset'):
    print(f"\n{'='*70}")
    print(f"  {dataset_name}")
    print(f"{'='*70}")
    print(group.drop(columns='Dataset').to_string(index=False))


  Bursty
Configuration  GPU mem-time (GB·s)  Mean resident (MB)  Peak (MB)  GPU mem reduction (%)  Mean mem reduction (%)
    Openwhisk             98869.66             7030.24    22910.0                    0.0                     0.0
       Unlock             39163.37             2784.76    10976.0                   60.4                    60.4
      Process             30161.84             2144.69    10976.0                   69.5                    69.5
     Profiler             30084.04             2139.16    10976.0                   69.6                    69.6
         NMIG               529.87               37.68    11071.0                   99.5                    99.5

  Normal
Configuration  GPU mem-time (GB·s)  Mean resident (MB)  Peak (MB)  GPU mem reduction (%)  Mean mem reduction (%)
    Openwhisk             44965.13             3197.30    13254.0                    0.0                     0.0
       Unlock             34425.18             2447.84     4067.0           

/tmp/ipykernel_3834051/2579614962.py:81: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_results = df_results.groupby('Dataset', group_keys=False).apply(add_reductions)
